In [1]:
import pennylane as qml
import numpy as np
import random

# 매 라운드마다 동일한 correlated error 발생하는 상황 (매 라운드에 동일하게 적용되는 확률 설정 가능)

## Setting

In [2]:
n_data = 9
n_anc = 8
n_qubits = n_data + n_anc

data_qubits = list(range(n_data))
anc_z = list(range(n_data, n_data + 4))
anc_x = list(range(n_data + 4, n_qubits))

DATA_WIRES = data_qubits
ANC_WIRES = anc_z + anc_x

## Pauli Operator and Two_qubit pauli error

In [3]:
# --------------------------------------------------
# Basic Pauli application
# --------------------------------------------------

def apply_pauli(pauli_type, wires):
    if pauli_type == 'X':
        qml.PauliX(wires=wires)
    elif pauli_type == 'Z':
        qml.PauliZ(wires=wires)
    elif pauli_type == 'Y':
        qml.PauliY(wires=wires)


def apply_two_qubit_pauli_error(error_list, error_wires):
    """
    error_list 예: ['X', 'Z']
    error_wires 예: [0, 9]
    """
    for p, w in zip(error_list, error_wires):
        apply_pauli(p, wires=w)

## Stabilizer Measurement

In [4]:
# 각 stabilizer가 어떤 data qubit에 연결되는지
X_stabilizers = [
    [0,1],
    [1,2,4,5],
    [3,4,6,7],
    [7,8]
]

Z_stabilizers = [
    [0,1,3,4],
    [2,5],
    [3,6],
    [4,5,7,8],
]

def measure_Z_stabilizer(anc, qubits):
    for q in qubits:
        qml.CNOT(wires=[q, anc])

def measure_X_stabilizer(anc, qubits):
    qml.Hadamard(wires=anc)
    for q in qubits:
        qml.CNOT(wires=[anc, q])
    qml.Hadamard(wires=anc)

# --------------------------------------------------
# One stabilizer measurement with ancilla reset
# --------------------------------------------------

def measure_Z_stabilizer_with_noise_and_reset(
    anc,
    qubits,
    error_list=[],
    error_wires=[],
    error_prob=0.0,
):
    """
    Z stabilizer 측정.

    네 기존 코드와 동일하게:
        data q -> anc 로 CNOT
    """

    sorted_error_wires = sorted(error_wires)

    for q in qubits:
        qml.CNOT(wires=[q, anc])

        sorted_wires = sorted([q, anc])

        if sorted_wires == sorted_error_wires:
            if random.random() < error_prob:
                apply_two_qubit_pauli_error(error_list, error_wires)

    m = qml.measure(anc, reset=True)
    return m


def measure_X_stabilizer_with_noise_and_reset(
    anc,
    qubits,
    error_list=[],
    error_wires=[],
    error_prob=0.0,
):
    """
    X stabilizer 측정.

    네 기존 코드와 동일하게:
        H on anc
        anc -> data q 로 CNOT
        H on anc
    """

    sorted_error_wires = sorted(error_wires)

    qml.Hadamard(wires=anc)

    for q in qubits:
        qml.CNOT(wires=[anc, q])

        sorted_wires = sorted([anc, q])

        if sorted_wires == sorted_error_wires:
            if random.random() < error_prob:
                apply_two_qubit_pauli_error(error_list, error_wires)

    qml.Hadamard(wires=anc)

    m = qml.measure(anc, reset=True)
    return m


## Logical 0 state preparation

In [6]:
def prepare_logical_zero():
    """
    Prepare |0_L> of the d=3 rotated surface code
    with logical Z_L = Z0 Z1 Z2.
    """

    # Independent variables:
    # a -> qubit 0
    # b -> qubit 2
    # c -> qubit 3
    # d -> qubit 8
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=2)
    qml.Hadamard(wires=3)
    qml.Hadamard(wires=8)

    # q1 = a ⊕ b
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[2, 1])

    # q4 = b ⊕ c
    qml.CNOT(wires=[2, 4])
    qml.CNOT(wires=[3, 4])

    # q5 = b
    qml.CNOT(wires=[2, 5])

    # q6 = c
    qml.CNOT(wires=[3, 6])

    # q7 = c ⊕ d
    qml.CNOT(wires=[3, 7])
    qml.CNOT(wires=[8, 7])

## Syndrome Measure

In [5]:
# --------------------------------------------------
# One full syndrome extraction round
# --------------------------------------------------

def syndrome_round_with_noise(
    error_list=[],
    error_wires=[],
    error_prob=0.0,
):
    """
    한 round에서 8개 syndrome 측정.

    순서:
        Z stabilizer 4개
        X stabilizer 4개

    반환:
        [z0, z1, z2, z3, x0, x1, x2, x3]
    """

    round_record = []

    for i, stab in enumerate(Z_stabilizers):
        m = measure_Z_stabilizer_with_noise_and_reset(
            anc=anc_z[i],
            qubits=stab,
            error_list=error_list,
            error_wires=error_wires,
            error_prob=error_prob,
        )
        round_record.append(m)

    for i, stab in enumerate(X_stabilizers):
        m = measure_X_stabilizer_with_noise_and_reset(
            anc=anc_x[i],
            qubits=stab,
            error_list=error_list,
            error_wires=error_wires,
            error_prob=error_prob,
        )
        round_record.append(m)

    return round_record

In [19]:
# --------------------------------------------------
# Repeated syndrome measurement QNode
# --------------------------------------------------

shots = 10
n_rounds = 5

@qml.qnode(qml.device("default.qubit", wires=n_qubits, shots=shots))
def repeated_syndrome_measurement(
    error_list=[],
    error_wires=[],
    error_prob=0.0,
    initial_error_list=[],
    initial_error_wires=[],
):
    measurement_record = []

    # logical |0_L> 준비
    prepare_logical_zero()

    qml.Barrier(wires=DATA_WIRES)

    # 초기 data error 또는 원하는 위치의 fixed error 삽입
    for p, w in zip(initial_error_list, initial_error_wires):
        apply_pauli(p, wires=w)

    # 반복 syndrome extraction
    for t in range(n_rounds):
        round_record = syndrome_round_with_noise(
            error_list=error_list,
            error_wires=error_wires,
            error_prob=error_prob,
        )

        measurement_record.extend(round_record)
        qml.Barrier()

    return [qml.sample(m) for m in measurement_record]

/Users/jhan/Library/Mobile Documents/com~apple~CloudDocs/ETRI/연구/Correlated Noise Estimation by Syndrome Measurement/syndrome_env/lib/python3.14/site-packages/pennylane/devices/device_api.py:201: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


In [26]:
qml.draw_mpl(repeated_syndrome_measurement, show_all_wires=True)(
    error_list=['X', 'Z'],
    error_wires=[0, 9],
    error_prob=0.5,
    initial_error_list=[],
    initial_error_wires=[],
)

(<Figure size 26000x2850 with 1 Axes>, <Axes: >)

In [21]:

# --------------------------------------------------
# Run and reshape
# --------------------------------------------------


measurement_output = repeated_syndrome_measurement(
    error_list=['X', 'Z'],
    error_wires=[0, 9],
    error_prob=1.0,
)

raw = np.array(measurement_output)

print("raw shape:", raw.shape)
# expected: (n_rounds * 8, shots)

syndrome = raw.reshape(n_rounds, 8, shots)
syndrome = np.moveaxis(syndrome, 2, 0)

print("syndrome shape:", syndrome.shape)
# expected: (shots, n_rounds, 8)

# syndrome[shot, round, stabilizer_index]
print(syndrome[0])

raw shape: (40, 10)
syndrome shape: (10, 5, 8)
[[0 0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]]


In [25]:
syndrome.shape

(10, 5, 8)

## Round마다 error 발생 여부 설정 가능하도록

In [ ]:
import pennylane as qml
import numpy as np

# --------------------------------------------------
# Pauli error utilities
# --------------------------------------------------

def apply_pauli(pauli_type, wires):
    if pauli_type == 'X':
        qml.PauliX(wires=wires)
    elif pauli_type == 'Y':
        qml.PauliY(wires=wires)
    elif pauli_type == 'Z':
        qml.PauliZ(wires=wires)


def apply_two_qubit_pauli_error(error_list, error_wires):
    for p, w in zip(error_list, error_wires):
        apply_pauli(p, wires=w)


def should_inject_fault(current_round, fault_round, gate_wires, fault_wires):
    return (
        current_round == fault_round
        and sorted(gate_wires) == sorted(fault_wires)
    )


# --------------------------------------------------
# Single stabilizer measurements with reset
# --------------------------------------------------

def measure_Z_stabilizer_single_fault(
    anc,
    qubits,
    current_round,
    fault_round=None,
    fault_wires=[],
    fault_error=[],
):
    """
    Z stabilizer 측정.

    기존 코드와 동일:
        data q -> ancilla CNOT

    해당 round, 해당 CNOT 위치에서만 fault_error 삽입.
    """

    for q in qubits:
        qml.CNOT(wires=[q, anc])

        if should_inject_fault(
            current_round=current_round,
            fault_round=fault_round,
            gate_wires=[q, anc],
            fault_wires=fault_wires,
        ):
            apply_two_qubit_pauli_error(fault_error, fault_wires)

    m = qml.measure(anc, reset=True)
    return m


def measure_X_stabilizer_single_fault(
    anc,
    qubits,
    current_round,
    fault_round=None,
    fault_wires=[],
    fault_error=[],
):
    """
    X stabilizer 측정.

    기존 코드와 동일:
        H on anc
        ancilla -> data q CNOT
        H on anc

    해당 round, 해당 CNOT 위치에서만 fault_error 삽입.
    """

    qml.Hadamard(wires=anc)

    for q in qubits:
        qml.CNOT(wires=[anc, q])

        if should_inject_fault(
            current_round=current_round,
            fault_round=fault_round,
            gate_wires=[anc, q],
            fault_wires=fault_wires,
        ):
            apply_two_qubit_pauli_error(fault_error, fault_wires)

    qml.Hadamard(wires=anc)

    m = qml.measure(anc, reset=True)
    return m


# --------------------------------------------------
# One full syndrome extraction round
# --------------------------------------------------

def syndrome_round_single_fault(
    current_round,
    fault_round=None,
    fault_wires=[],
    fault_error=[],
):
    """
    한 round에서 8개 stabilizer 측정.

    반환 순서:
        [Z0, Z1, Z2, Z3, X0, X1, X2, X3]
    """

    round_record = []

    for i, stab in enumerate(Z_stabilizers):
        m = measure_Z_stabilizer_single_fault(
            anc=anc_z[i],
            qubits=stab,
            current_round=current_round,
            fault_round=fault_round,
            fault_wires=fault_wires,
            fault_error=fault_error,
        )
        round_record.append(m)

    for i, stab in enumerate(X_stabilizers):
        m = measure_X_stabilizer_single_fault(
            anc=anc_x[i],
            qubits=stab,
            current_round=current_round,
            fault_round=fault_round,
            fault_wires=fault_wires,
            fault_error=fault_error,
        )
        round_record.append(m)

    return round_record


# --------------------------------------------------
# Repeated syndrome measurement QNode
# --------------------------------------------------

def make_single_fault_repeated_syndrome_qnode(n_rounds, shots=1000):
    dev = qml.device("default.qubit", wires=n_qubits, shots=shots)

    @qml.qnode(dev)
    def repeated_syndrome_single_fault(
        fault_round=None,
        fault_wires=[],
        fault_error=[],
        initial_error_wires=[],
        initial_error_list=[],
    ):
        measurement_record = []

        # logical |0_L> 준비
        prepare_logical_zero()

        qml.Barrier(wires=DATA_WIRES)

        # 선택적 초기 data error
        for p, w in zip(initial_error_list, initial_error_wires):
            apply_pauli(p, wires=w)

        # repeated syndrome extraction
        for t in range(n_rounds):
            round_record = syndrome_round_single_fault(
                current_round=t,
                fault_round=fault_round,
                fault_wires=fault_wires,
                fault_error=fault_error,
            )
            measurement_record.extend(round_record)

        return [qml.sample(m) for m in measurement_record]

    return repeated_syndrome_single_fault


# --------------------------------------------------
# Example usage
# --------------------------------------------------

n_rounds = 4
shots = 1000

single_fault_qnode = make_single_fault_repeated_syndrome_qnode(
    n_rounds=n_rounds,
    shots=shots,
)

raw = single_fault_qnode(
    fault_round=1,
    fault_wires=[0, 9],
    fault_error=['X', 'Z'],
)

raw = np.array(raw)

print("raw shape:", raw.shape)
# expected: (n_rounds * 8, shots)

syndrome = raw.reshape(n_rounds, 8, shots)
syndrome = np.moveaxis(syndrome, 2, 0)

print("syndrome shape:", syndrome.shape)
# expected: (shots, n_rounds, 8)

detection_events = syndrome[:, 1:, :] ^ syndrome[:, :-1, :]

print("detection_events shape:", detection_events.shape)
# expected: (shots, n_rounds - 1, 8)

print("first shot syndrome:")
print(syndrome[0])

print("first shot detection events:")
print(detection_events[0])